In [1]:
%pip install -U spacy

Note: you may need to restart the kernel to use updated packages.


In [5]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_md")

In [29]:
text = "The quick brown fox jumps over the lazy dog"

doc0 = nlp(text)

2.2 Trực quan hóa cây phụ thuộc

In [ ]:
# displacy.serve(doc, style='dep')
displacy.render(doc0)

Câu hỏi:
- Từ nào là gốc ROOT của câu?
    - Từ gốc cỏa câu là jumps (VERB)
- jumps có những từ phụ thuộc nào? Các quan hệ đó là gì?
    - jumps các từ phụ thuộc là fox(NOUN với quan hệ nsubj) và over(ADP với quan hệ prep)
- fox là head của những từ nào?
    - fox là head của The, quick, brown

Phần 3: Truy cập các thành phần trong cây phụ thuộc

In [25]:
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)

print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)

for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------
Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | compound   | startup      | NOUN     | []
startup      | dobj       | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | NOUN     | ['billion']
$            | quantmod   | billion      | NUM      | []
1            | compound   | billion      | NUM      | []
billion      | pobj       | for          | ADP      | ['$', '1']


Phần 4: Duyệt cây phụ thuộc để trích xuất thông tin

4.1. Bài toán: Tìm chủ ngữ và tân ngữ của một động từ

In [20]:
text = "The cat chased the mouse and the dog watched them."
doc1 = nlp(text)
for token in doc1:
# Chỉ tìm các động từ
    if token.pos_ == "VERB":
        verb = token.text
        subject = ""
    obj = ""
    # Tìm chủ ngữ (nsubj) và tân ngữ (dobj) trong các con của động từ
    for child in token.children:
        if child.dep_ == "nsubj":
            subject = child.text
        if child.dep_ == "dobj":
            obj = child.text
    if subject and obj:
        print(f"Found Triplet: ({subject}, {verb}, {obj})")

Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


4.2. Bài toán: Tìm các tính từ bổ nghĩa cho một danh từ

In [21]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc2 = nlp(text)
for token in doc2:
    # Chỉ tìm các danh từ
    if token.pos_ == "NOUN":
        adjectives = []
        # Tìm các tính từ bổ nghĩa (amod) trong các con của danh từ
        for child in token.children:
            if child.dep_ == "amod":
                adjectives.append(child.text)
        if adjectives:
            print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")

Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


Phần 5: Bài tập tự luyện

Bài 1: Tìm động từ chính của câu

Động từ chính của câu thường có quan hệ ROOT. Viết một hàm
find_main_verb(doc) nhận vào một đối tượng Doc của spaCy và trả về
Token là động từ chính.

In [26]:
def find_main_verb(doc):
    for token in doc:
        if token.dep_ == "ROOT":
            return token.text

print(find_main_verb(doc1))
print(find_main_verb(doc2))
print(find_main_verb(doc))

chased
sleeping
looking


Bài 2: Trích xuất các cụm danh từ (Noun Chunks)

spaCy đã có sẵn thuộc tính .noun_chunks để trích xuất các cụm danh
từ. Tuy nhiên, hãy thử tự viết một hàm để làm điều tương tự.

Gợi ý: Một cụm danh từ đơn giản là một danh từ và tất cả các từ bổ
nghĩa cho nó (như det, amod, compound). Bạn có thể bắt đầu từ một
danh từ và duyệt ngược lên head hoặc duyệt xuống các children của
nó.

In [34]:
def find_noun_chunks(doc):
    chunks = []
    for token in doc:
        if token.pos_ == "NOUN":
            chunk = []
            chunk.append(token.text)
            for child in token.children:
                chunk.append(child.text)
            chunks.append(" ".join(chunk))
    return chunks

find_noun_chunks(doc)
find_noun_chunks(doc0)

['fox The quick brown', 'dog the lazy']

Bài 3: Tìm đường đi ngắn nhất trong cây

Viết một hàm get_path_to_root(token) để tìm đường đi từ một token
bất kỳ lên đến gốc (ROOT) của cây. Hàm nên trả về một danh sách
các token trên đường đi.

In [37]:
def get_path_to_root(token):
    path = [token.text]
    while token.dep_ != "ROOT":
        token = token.head
        path.append(token.text)
    return path

for token in doc0:
    print(get_path_to_root(token))

['The', 'fox', 'jumps']
['quick', 'fox', 'jumps']
['brown', 'fox', 'jumps']
['fox', 'jumps']
['jumps']
['over', 'jumps']
['the', 'dog', 'over', 'jumps']
['lazy', 'dog', 'over', 'jumps']
['dog', 'over', 'jumps']
